In [2]:
import pandas as pd
import re

# 1. ЗАГРУЗКА ДАТАСЕТА
df = pd.read_csv('/kaggle/input/datasets/asan104/newsss/synthetic_text_data.csv')

print(f"Исходный размер: {df.shape}")

# 2. ВЫБОР ТЕКСТОВОЙ КОЛОНКИ
text_column = 'text'  # <-- замените на имя вашей колонки

# Убираем пропуски
df = df.dropna(subset=[text_column])

# 3. УДАЛЕНИЕ СПЕЦСИМВОЛОВ (включая точки и запятые)
def remove_special_chars(text):
    """
    Оставляет только буквы (русские и латинские), цифры и пробелы.
    Точки, запятые и все прочие знаки препинания удаляются.
    """
    # Разрешены: буквы, цифры, пробелы. Всё остальное (в т.ч. . и ,) — удаляется
    cleaned = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s]', '', str(text))
    # Убираем множественные пробелы
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

df[text_column] = df[text_column].apply(remove_special_chars)

# 4. УДАЛЕНИЕ СТРОК КОРОЧЕ 50 СИМВОЛОВ
df = df[df[text_column].str.len() >= 50]

print(f"После удаления коротких строк: {df.shape}")

# 5. УДАЛЕНИЕ ДУБЛИКАТОВ
df = df.drop_duplicates(subset=[text_column])

print(f"После удаления дубликатов: {df.shape}")

# 6. СОХРАНЕНИЕ РЕЗУЛЬТАТА
df.to_csv('/kaggle/working/clean.csv', index=False)

print("Готово! Файл сохранён в /kaggle/working/clean.csv")




Исходный размер: (500, 2)
После удаления коротких строк: (500, 2)
После удаления дубликатов: (386, 2)
Готово! Файл сохранён в /kaggle/working/clean.csv


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Загрузка
df = pd.read_csv("/kaggle/working/clean.csv")
print("Колонки:", df.columns.tolist())
print(df.head())

# 2. TF-IDF
docs = df["text"].astype(str).tolist()
vec = TfidfVectorizer(max_features=1000)
X = vec.fit_transform(docs)

# 3. Проверка
print("Размер матрицы:", X.shape)
print("Словарь:", vec.get_feature_names_out()[:20])
print(X.toarray()[:3])

# 4. Сохранение
tfidf_df = pd.DataFrame(X.toarray(),
                        columns=vec.get_feature_names_out())
tfidf_df.to_csv("/kaggle/working/tfidf.csv", index=False)
print("Сохранено: tfidf.csv, размер =", tfidf_df.shape)

Колонки: ['text', 'label']
                                                text       label
0  A recent study shows that nutritionists recomm...      Health
1  According to reports cybersecurity specialists...  Technology
2  public sources state that a severe category fo...     Weather
3  Officials announced that the government implem...    Politics
4  Experts note that formula one teams are introd...      Sports
Размер матрицы: (386, 571)
Словарь: ['accelerate' 'access' 'according' 'across' 'action' 'adaptation'
 'address' 'addressed' 'adopting' 'advanced' 'advancements' 'aerodynamic'
 'after' 'ahead' 'airline' 'airports' 'album' 'algorithms' 'alliances'
 'along']
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.19355491 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
Сохранено: tfidf.csv, размер = (386, 571)


In [9]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Предполагается, что tfidf.csv уже сохранён (задание №3)
tfidf_df = pd.read_csv('/kaggle/working/tfidf.csv')
df = pd.read_csv('/kaggle/working/clean.csv')

# Метки
le = LabelEncoder()
y = le.fit_transform(df['label'])
n_classes = len(le.classes_)
print("Классов:", n_classes, "→", list(le.classes_))

# Train/test
X_train, X_test, y_train, y_test = train_test_split(
    tfidf_df.values, y, test_size=0.2, random_state=42, stratify=y)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)

# Модель
class Classifier(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.fc1 = nn.Linear(n_in, n_hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = Classifier(n_in=tfidf_df.shape[1], n_hidden=128, n_out=n_classes)
print(model)

# Обучение
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train), y_train)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

# Оценка
model.eval()
with torch.no_grad():
    pred = model(X_test).argmax(dim=1)
    print("Accuracy:", round((pred == y_test).float().mean().item(), 4))

Классов: 7 → ['Business', 'Entertainment', 'Health', 'Politics', 'Sports', 'Technology', 'Weather']
Classifier(
  (fc1): Linear(in_features=571, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)
Epoch 5: loss = 1.9165
Epoch 10: loss = 1.8721
Epoch 15: loss = 1.8160
Epoch 20: loss = 1.7440
Accuracy: 1.0
